# 05 — LoRA Fine-Tuning

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/winstonsmith1897/DantinoX/blob/main/docs/notebooks/05_lora_fine_tuning.ipynb)

This notebook fine-tunes a pretrained DantinoX AR checkpoint using **Low-Rank Adaptation (LoRA)**.

**Key concepts:**
- `use_lora=True` injects rank-decomposed adapter matrices into attention Q/K/V/O projections
- Only adapter weights are updated; base model weights are frozen
- `lora_rank` (r) controls adapter capacity, `lora_alpha` controls the effective LR scaling
- The adapter adds `2 × r × d` parameters per projection — typically 0.5–2% of total

In [ ]:
!pip install -q git+https://github.com/winstonsmith1897/DantinoX.git#egg=dantinox[all]

## 1. Pretrain a base model

We first train a small AR model on Tiny Shakespeare for 2 epochs, then fine-tune it on a different text.

In [ ]:
import urllib.request, os
if not os.path.exists('tiny_shakespeare.txt'):
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt',
        'tiny_shakespeare.txt'
    )
if not os.path.exists('bible.txt'):
    # King James Bible as fine-tuning target domain
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/mxw/grmr/master/src/finaltests/bible.txt',
        'bible.txt'
    )
print('Data ready.')

In [ ]:
import dantinox as dx

base_run = dx.fit(
    'ar', 'tiny_shakespeare.txt',
    dim=256, n_heads=4, head_size=64, num_blocks=4, vocab_size=200,
    lr=3e-4, epochs=2, batch_size=16,
)
print('Base model checkpoint:', base_run)

## 2. LoRA configuration

LoRA replaces each target weight matrix W with `W + (α/r) · B·A`, where:
- A ∈ ℝ^(r×d_in) — low-rank down-projection (random Gaussian init)
- B ∈ ℝ^(d_out×r) — low-rank up-projection (zero init, so adapter = 0 at start)

Setting `lora_targets='attention'` injects adapters into all four attention projections (Q, K, V, O). During training only A and B are updated.

In [ ]:
from dantinox.core.config import Config
from dantinox.core.model import Transformer
from dantinox.core.lora import LoRAParam
from dantinox.training.trainer import _msgpack_load
from flax import nnx
import yaml, jax

# Load base model config
with open(f'{base_run}/config.yaml') as f:
    raw = yaml.safe_load(f)
cfg = Config.from_dict(raw)

# Re-init with LoRA enabled
lora_cfg = Config.from_dict({**raw,
    'use_lora': True,
    'lora_rank': 8,
    'lora_alpha': 16.0,
    'lora_targets': 'attention',
    'lora_dropout': 0.05,
})

rngs  = nnx.Rngs(42)
model = Transformer(lora_cfg, rngs=rngs)

# Load pretrained base weights into the LoRA model
ckpt_path = f'{base_run}/checkpoint_best.msgpack'
raw_ckpt = _msgpack_load(ckpt_path)
state = nnx.state(model, nnx.Not(nnx.RngState))
state.replace_by_pure_dict(raw_ckpt)
nnx.update(model, state)

# Count base vs. adapter parameters
all_params   = nnx.state(model, nnx.Param)
lora_params  = nnx.state(model, LoRAParam)
base_count   = sum(x.size for x in jax.tree_util.tree_leaves(all_params))
lora_count   = sum(x.size for x in jax.tree_util.tree_leaves(lora_params))
print(f'Base parameters  : {base_count:,}')
print(f'LoRA parameters  : {lora_count:,}  ({100*lora_count/base_count:.2f}% of total)')
print(f'Trainable ratio  : {lora_count}/{base_count} = {lora_count/base_count:.4f}')

## 3. Fine-tuning

The `Trainer` automatically freezes base model weights when `use_lora=True`. Only the LoRA adapter matrices are updated via the optimizer.

In [ ]:
import dataclasses

# Build an AR paradigm with LoRA adapters on attention projections
lora_model_cfg = dx.ModelConfig(
    dim=256, n_heads=4, head_size=64, num_blocks=4, vocab_size=200, causal=True,
    use_lora=True,
    lora_rank=8,
    lora_alpha=16.0,
    lora_targets='attention',
    lora_dropout=0.05,
)

ft_config = dx.TrainingConfig(
    lr         = 1e-3,
    epochs     = 2,
    batch_size = 16,
)

lora_paradigm = dx.ARParadigm(lora_model_cfg)
ft_run = dx.Trainer(lora_paradigm, ft_config).fit('bible.txt')
print('Fine-tuned checkpoint:', ft_run)

## 4. Compare base vs fine-tuned generation

In [ ]:
prompt = 'In the beginning'

print('=== Base model ===')
base_out = dx.quick_generate(base_run, prompt, max_new_tokens=100)
print(base_out)

print('\n=== Fine-tuned (LoRA) ===')
ft_out = dx.quick_generate(ft_run, prompt, max_new_tokens=100)
print(ft_out)

## 5. Merging adapters into the base model

After fine-tuning you can merge the LoRA matrices back into the base weights for deployment. The merged model is identical in architecture to the original — no LoRA overhead at inference.

In [ ]:
from dantinox.core.lora import merge_lora

# merge_lora folds W_lora = (alpha/rank) * B @ A into W
merged_model = merge_lora(model)

# Verify parameter counts match the base model
merged_params = sum(x.size for x in jax.tree_util.tree_leaves(nnx.state(merged_model, nnx.Param)))
print(f'Merged model parameters: {merged_params:,}  (same as base: {base_count:,})')

## 6. Rank ablation

LoRA rank `r` trades off capacity vs. parameter efficiency. Lower ranks are faster to train and less likely to overfit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from dantinox.core.config import Config
from dantinox.core.model import Transformer

ranks   = [2, 4, 8, 16, 32]
n_lora  = []
pct     = []

for r in ranks:
    cfg_r = Config.from_dict({**raw, 'use_lora': True, 'lora_rank': r, 'lora_alpha': 2*r})
    m_r   = Transformer(cfg_r, rngs=nnx.Rngs(0))
    lp    = nnx.state(m_r, LoRAParam)
    lc    = sum(x.size for x in jax.tree_util.tree_leaves(lp))
    n_lora.append(lc)
    pct.append(100 * lc / base_count)

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(range(len(ranks)), pct, color='steelblue', alpha=0.8)
ax.set_xticks(range(len(ranks)))
ax.set_xticklabels([f'r={r}' for r in ranks])
ax.set_ylabel('LoRA params (% of total)')
ax.set_title('LoRA parameter count vs. rank')
for i, (p, n) in enumerate(zip(pct, n_lora)):
    ax.text(i, p + 0.03, f'{n:,}', ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.show()